# Assignment 7 — Customer Segmentation using K-Means Clustering and PCA

**Dataset:** [Mall Customer Segmentation Dataset (Kaggle)](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

> Download `Mall_Customers.csv` from the Kaggle link above and place it in the same folder as this notebook before running.

In [1]:
# ============================================================
# Assignment 7 - Customer Segmentation using K-Means Clustering
# and Principal Component Analysis (PCA)
# ============================================================
# Dataset: Mall Customer Segmentation Dataset (Kaggle)
# https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
#
# NOTE: Download "Mall_Customers.csv" from the Kaggle link above and
# place it in the same folder as this script before running.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)

## TASK 1: DATA UNDERSTANDING 

In [2]:
print("=" * 60)
print("TASK 1: DATA UNDERSTANDING")
print("=" * 60)

df = pd.read_csv("Mall_Customers.csv")

print("\nFirst five records:")
print(df.head())

numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

print(f"\nNumerical features   : {numerical_features}")
print(f"Categorical features : {categorical_features}")

print("\nDataset Info:")
df.info()

print("\nSummary Statistics:")
print(df.describe(include="all"))

TASK 1: DATA UNDERSTANDING

First five records:
   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40

Numerical features   : ['CustomerID', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
Categorical features : ['Gender']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CustomerID              200 non-null    int64 
 1   Gender                  200 non-null    object
 2   Age                     200 non-null    int64 
 3   Annual Income 

## TASK 2: DATA PREPROCESSING

In [4]:
print("\n" + "=" * 60)
print("TASK 2: DATA PREPROCESSING")
print("=" * 60)

print("\nMissing values per column:")
print(df.isnull().sum())

# Remove unnecessary columns (CustomerID is just an identifier)
df_clean = df.drop(columns=["CustomerID"])

# Encode categorical variable (Genre: Male/Female)
le = LabelEncoder()
df_clean["Gender"] = le.fit_transform(df_clean["Gender"])  # Male=1, Female=0 (alphabetical)
print(f"\nEncoded 'Genre' classes: {list(le.classes_)} -> {list(le.transform(le.classes_))}")

print("\nCleaned dataframe preview:")
print(df_clean.head())

# Standardize numerical features
features_for_clustering = ["Annual Income (k$)", "Spending Score (1-100)"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[features_for_clustering])

print(f"\nFeatures used for clustering: {features_for_clustering}")
print("Scaled feature sample (first 5 rows):")
print(X_scaled[:5])


TASK 2: DATA PREPROCESSING

Missing values per column:
CustomerID                0
Gender                    0
Age                       0
Annual Income (k$)        0
Spending Score (1-100)    0
dtype: int64

Encoded 'Genre' classes: ['Female', 'Male'] -> [np.int64(0), np.int64(1)]

Cleaned dataframe preview:
   Gender  Age  Annual Income (k$)  Spending Score (1-100)
0       1   19                  15                      39
1       1   21                  15                      81
2       0   20                  16                       6
3       0   23                  16                      77
4       0   31                  17                      40

Features used for clustering: ['Annual Income (k$)', 'Spending Score (1-100)']
Scaled feature sample (first 5 rows):
[[-1.73899919 -0.43480148]
 [-1.73899919  1.19570407]
 [-1.70082976 -1.71591298]
 [-1.70082976  1.04041783]
 [-1.66266033 -0.39597992]]


## TASK 3: MODEL DEVELOPMENT


In [5]:
print("\n" + "=" * 60)
print("TASK 3: MODEL DEVELOPMENT")
print("=" * 60)

# --- Elbow Method ---
inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure()
plt.plot(list(K_range), inertia, marker="o", linewidth=2)
plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.xticks(list(K_range))
plt.tight_layout()
plt.savefig("elbow_curve.png", dpi=150)
plt.close()
print("Saved: elbow_curve.png")

# Optimal K chosen from the elbow curve
optimal_k = 5

# --- Train final K-Means model ---
kmeans = KMeans(n_clusters=optimal_k, init="k-means++", random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)
df_clean["Cluster"] = cluster_labels

print(f"\nOptimal number of clusters chosen: {optimal_k}")
print("\nCluster label counts:")
print(df_clean["Cluster"].value_counts().sort_index())

# --- PCA to 2 components ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_clean["PC1"] = X_pca[:, 0]
df_clean["PC2"] = X_pca[:, 1]

print(f"\nExplained variance ratio by 2 PCA components: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")


TASK 3: MODEL DEVELOPMENT
Saved: elbow_curve.png

Optimal number of clusters chosen: 5

Cluster label counts:
Cluster
0    81
1    39
2    22
3    35
4    23
Name: count, dtype: int64

Explained variance ratio by 2 PCA components: [0.50495142 0.49504858]
Total variance explained: 100.00%


## TASK 4: VISUALIZATION AND EVALUATION 

In [6]:
print("\n" + "=" * 60)
print("TASK 4: VISUALIZATION AND EVALUATION")
print("=" * 60)

# Scatter plot of clusters on original feature space
plt.figure()
palette = sns.color_palette("Set2", optimal_k)
for c in range(optimal_k):
    subset = df_clean[df_clean["Cluster"] == c]
    plt.scatter(subset["Annual Income (k$)"], subset["Spending Score (1-100)"],
                s=60, color=palette[c], label=f"Cluster {c}")
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers_original[:, 0], centers_original[:, 1],
            s=250, c="black", marker="X", label="Centroids")
plt.title("Customer Segments (Annual Income vs Spending Score)")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.tight_layout()
plt.savefig("cluster_scatter.png", dpi=150)
plt.close()
print("Saved: cluster_scatter.png")

# PCA visualization with cluster labels
plt.figure()
for c in range(optimal_k):
    subset = df_clean[df_clean["Cluster"] == c]
    plt.scatter(subset["PC1"], subset["PC2"], s=60, color=palette[c], label=f"Cluster {c}")
plt.title("PCA Visualization of Customer Clusters (2 Components)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.tight_layout()
plt.savefig("pca_visualization.png", dpi=150)
plt.close()
print("Saved: pca_visualization.png")

# --- Cluster profiling ---
print("\nCluster-wise mean characteristics:")
cluster_summary = df_clean.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
print(cluster_summary)

print("""
Observations:
1. Optimal number of clusters: The elbow curve shows inertia dropping sharply
   up to K=5, after which the decrease flattens out considerably. This "elbow"
   at K=5 indicates 5 is the most natural number of customer segments for this
   data (matching the classic Income vs Spending Score pattern).
2. Role of PCA: The original scaled feature space is only 2-dimensional here
   (Income, Spending Score), so PCA mainly serves to demonstrate dimensionality
   reduction and produce an uncorrelated 2D projection that preserves nearly
   all the variance (see explained_variance_ratio_ above). In datasets with
   more numeric features (e.g. including Age), PCA would be essential to
   visualize clusters that exist in a higher-dimensional space.
3. Customer group characteristics: The five clusters correspond to distinct,
   business-relevant segments: (a) low income / low spending - budget-conscious
   shoppers, (b) low income / high spending - value-seeking spenders, (c) mid
   income / mid spending - average/typical customers, (d) high income / low
   spending - cautious high-earners who are under-targeted, and (e) high
   income / high spending - the most valuable "premium" customer segment.
4. Cluster separation: Clusters are visually well-separated in both the
   original feature scatter plot and the PCA projection, confirming that
   K-Means found meaningful, non-overlapping groupings.
""")


TASK 4: VISUALIZATION AND EVALUATION
Saved: cluster_scatter.png
Saved: pca_visualization.png

Cluster-wise mean characteristics:
          Age  Annual Income (k$)  Spending Score (1-100)
Cluster                                                  
0        42.7                55.3                    49.5
1        32.7                86.5                    82.1
2        25.3                25.7                    79.4
3        41.1                88.2                    17.1
4        45.2                26.3                    20.9

Observations:
1. Optimal number of clusters: The elbow curve shows inertia dropping sharply
   up to K=5, after which the decrease flattens out considerably. This "elbow"
   at K=5 indicates 5 is the most natural number of customer segments for this
   data (matching the classic Income vs Spending Score pattern).
2. Role of PCA: The original scaled feature space is only 2-dimensional here
   (Income, Spending Score), so PCA mainly serves to demonstrate dimens

## TASK 5: CONCLUSION

In [7]:
conclusion = """
TASK 5: CONCLUSION
============================================================
This project applied K-Means clustering to segment mall customers based on
their annual income and spending score, identifying five distinct groups
ranging from low-income budget shoppers to high-income premium spenders. The
Elbow Method confirmed five clusters as the optimal choice, and PCA was used
to project the data into two dimensions for clear visualization, showing
well-separated, meaningful segments. From a business perspective, these
segments enable the mall's management to design targeted marketing
campaigns - for example, loyalty rewards for high-income/high-spending
customers and promotional offers to convert high-income/low-spending
customers into active spenders. A key limitation of K-Means is that it
requires the number of clusters (K) to be specified in advance and assumes
spherical, similarly-sized clusters, which may not always reflect real
customer behavior. An advantage of PCA is that it reduces dimensionality
while retaining most of the data's variance, making complex, multi-feature
customer data easier to visualize and interpret without significant loss
of information.
"""
print(conclusion)

# Save cluster summary to CSV for reference
df_clean.to_csv("customer_segments_output.csv", index=False)
print("Saved: customer_segments_output.csv")
print("\nAll tasks completed successfully.")


TASK 5: CONCLUSION
This project applied K-Means clustering to segment mall customers based on
their annual income and spending score, identifying five distinct groups
ranging from low-income budget shoppers to high-income premium spenders. The
Elbow Method confirmed five clusters as the optimal choice, and PCA was used
to project the data into two dimensions for clear visualization, showing
well-separated, meaningful segments. From a business perspective, these
segments enable the mall's management to design targeted marketing
campaigns - for example, loyalty rewards for high-income/high-spending
customers and promotional offers to convert high-income/low-spending
customers into active spenders. A key limitation of K-Means is that it
requires the number of clusters (K) to be specified in advance and assumes
spherical, similarly-sized clusters, which may not always reflect real
customer behavior. An advantage of PCA is that it reduces dimensionality
while retaining most of the data's v